In [ ]:
# PREPROCESSING PIPELINE DOCUMENTATION
#
# OVERVIEW
# This pipeline reads one shared outdoor weather file and multiple indoor thermostat files,
# converts them into standardized event timelines, inserts synthetic expiration boundaries
# when data gaps exceed 30 minutes, propagates step-like states across time, interpolates
# temperature only within bounded valid windows, aligns outdoor conditions to each indoor
# event timestamp, converts event-to-event durations into operational runtime intervals,
# slices those intervals at calendar day boundaries, and aggregates the result into one
# daily feature row per equipment ID per day.
#
# -------------------------------------------------------------------------------
# 1. GLOBAL SETUP AND FILE DISCOVERY
# -------------------------------------------------------------------------------
# The notebook imports operating system, file discovery, numerical, dataframe, and
# machine learning libraries. It defines the base input folder, the output folder
# for processed daily files, the 30-minute threshold used throughout the timeline
# logic, the numeric reset token used to terminate numeric forward-fill behavior,
# the list of step-like indoor columns, and the list of columns that are considered
# leakage for later modeling stages.
#
# The output directory is created if it does not already exist.
#
# All CSV files under the base directory are discovered and sorted. The first file
# whose filename contains the word "outdoor" is treated as the single weather source.
# Every other CSV is treated as an indoor thermostat source.
#
# -------------------------------------------------------------------------------
# 2. OUTDOOR WEATHER PREPARATION
# -------------------------------------------------------------------------------
# The outdoor weather file is read using a semicolon delimiter.
#
# The Timestamp column is parsed into datetime form. Any row with an unparseable
# timestamp is removed. The remaining weather rows are sorted chronologically and
# their index is reset.
#
# The generic outdoor Temperature column is renamed to Outdoor_Temperature so that
# it does not collide with indoor temperature fields later in the pipeline.
#
# The required outdoor variables are:
# Outdoor_Temperature, outsideMinTemp, outsideMaxTemp, and outsideHumidity.
#
# Each of those required fields is explicitly converted to numeric form. Any value
# that cannot be interpreted numerically becomes missing.
#
# The resulting weather dataframe contains only Timestamp plus the four standardized
# outdoor variables.
#
# -------------------------------------------------------------------------------
# 3. INDOOR FILE INGESTION AND COLUMN STANDARDIZATION
# -------------------------------------------------------------------------------
# Each indoor thermostat file is read using a semicolon delimiter.
#
# Raw manufacturer-specific column names are mapped into canonical names:
# output_state becomes OutputState
# fan_state becomes FanState
# hvac_state or running_mode becomes RunningMode
# temp becomes Temperature
# set_point becomes Setpoint
#
# The Timestamp column is parsed into datetime form. Any row with an invalid
# timestamp is removed. The remaining rows are sorted by time and reindexed.
#
# Equipment_ID is derived from the filename and added as a column so that the
# processed data always carries the originating unit identifier.
#
# A raw_row_id column is added as a monotonic row counter. This is used later
# to preserve deterministic order when timestamps tie.
#
# -------------------------------------------------------------------------------
# 4. SAME-TIMESTAMP BURST RESOLUTION
# -------------------------------------------------------------------------------
# The code resolves cases where the device emitted multiple rows with the exact
# same timestamp.
#
# For every group of rows sharing one Timestamp, all non-timestamp columns are
# forward-filled downward only within that local timestamp group. This means that
# later rows inside the same burst inherit values already seen earlier within that
# same second.
#
# After this localized fill, only the last row from each timestamp group is kept.
# That retained last row represents the final resolved state snapshot for that
# timestamp after combining information from all rows in the burst.
#
# The index is reset after the burst reduction.
#
# -------------------------------------------------------------------------------
# 5. RUNNING MODE NORMALIZATION
# -------------------------------------------------------------------------------
# The raw running mode text is copied into RunningMode_raw for auditing.
#
# A normalization function converts running mode text into canonical categories:
# "heat" and "heating" become "heat"
# "cool" and "cooling" become "cool"
# "off", "idle", "none", "false", and "0" become "off"
# any other non-null text becomes "unknown"
# null input remains missing
#
# The normalized result is written to RunningMode_clean.
#
# -------------------------------------------------------------------------------
# 6. GAP DETECTION AND VIRTUAL EXPIRATION ROW INJECTION
# -------------------------------------------------------------------------------
# The code computes the time gap in seconds between each row and the immediately
# previous row using the Timestamp difference.
#
# Any row whose incoming gap exceeds 30 minutes is treated as the end of a blackout.
# For each such blackout, one synthetic row is created.
#
# The synthetic row is placed exactly 30 minutes after the last confirmed row before
# the gap. This timestamp is calculated by taking the start of the gap and adding
# the fixed 30-minute threshold.
#
# The synthetic row contains:
# the synthetic Timestamp
# the Equipment_ID copied from the surrounding data
# RunningMode_clean set to "unknown"
# Setpoint set to the numeric reset token
# FanState set to "unknown"
# is_virtual_expiration set to True
#
# These synthetic rows are appended to the timeline, then the whole dataframe is
# re-sorted chronologically and reindexed. The temporary gap_sec column is removed.
#
# Current limitation:
# Occupied is currently ignored by the virtual expiration logic. Even though Occupied
# appears in the declared list of step-like columns, the synthetic expiration row does
# not explicitly assign any Occupied reset value. As a result, Occupied is not actively
# terminated by the current blackout injection step.
#
# -------------------------------------------------------------------------------
# 7. UNBOUNDED STATE PERSISTENCE
# -------------------------------------------------------------------------------
# After virtual rows are inserted, RunningMode_clean is forward-filled down the full
# timeline to create the final RunningMode series.
#
# This means the normalized operating mode persists forward row by row until another
# real or synthetic row changes it.
#
# The same forward-fill process is applied to each step-like indoor column that is
# actually present in the dataframe:
# Setpoint, Mode, Occupied, RentalStatus, OutputState, FanState
#
# Before Setpoint is forward-filled, it is explicitly coerced to numeric form so that
# the numeric reset token can be propagated numerically.
#
# After Setpoint is forward-filled, every occurrence of the numeric reset token is
# replaced with missing. This converts the synthetic termination marker into a null
# setpoint region after the expiration boundary.
#
# The dataframe is then sorted again by Timestamp and raw_row_id using mergesort
# to preserve stable ordering when timestamps tie.
#
# -------------------------------------------------------------------------------
# 8. TEMPERATURE INTERPOLATION WITH TOTAL-GAP CHECK
# -------------------------------------------------------------------------------
# Indoor Temperature is not forward-filled. Instead, it is processed by a bounded
# time interpolation function.
#
# The target series is first coerced to numeric form. Non-numeric values become missing.
#
# For every row, the function identifies:
# the timestamp of the previous non-missing temperature value
# the timestamp of the next non-missing temperature value
#
# It then calculates the full width of that valid-to-valid bracket in minutes.
#
# A time-indexed interpolation is performed across the entire timestamp axis using
# pandas time interpolation.
#
# The interpolated result is only kept at positions where either:
# the original temperature was already present, or
# the total width from previous valid temperature to next valid temperature is
# less than or equal to 30 minutes
#
# If the total bracket width exceeds 30 minutes, the interpolated value is replaced
# with missing for that region.
#
# Synthetic expiration rows participate in the time axis because they have timestamps,
# but they do not count as valid temperature anchors unless they contain a non-missing
# temperature value.
#
# -------------------------------------------------------------------------------
# 9. INTERVAL RUNTIME ACCOUNTING
# -------------------------------------------------------------------------------
# The timeline is converted from event points into time intervals.
#
# prev_timestamp is created by shifting Timestamp downward by one row.
#
# dt_sec is computed as the number of seconds between the current Timestamp and
# prev_timestamp. The first row receives zero seconds.
#
# prior_mode is created by shifting RunningMode downward by one row so that each
# interval is credited to the mode that was active leading into the current timestamp.
# The first interval defaults to "unknown".
#
# Vectorized duration columns are then created:
# heat_sec receives dt_sec where prior_mode is "heat", otherwise zero
# cool_sec receives dt_sec where prior_mode is "cool", otherwise zero
# off_sec receives dt_sec where prior_mode is "off", otherwise zero
# unk_sec receives dt_sec where prior_mode is "unknown", otherwise zero
#
# -------------------------------------------------------------------------------
# 10. OUTDOOR WEATHER ALIGNMENT
# -------------------------------------------------------------------------------
# The weather attachment step aligns outdoor readings to each indoor event timestamp.
#
# A sorted copy of indoor timestamps is created.
#
# A backward as-of merge finds the most recent outdoor observation at or before each
# indoor timestamp. This provides:
# prev_outdoor_ts
# prev_outdoor_temp
# prev_outdoor_humidity
#
# A forward as-of merge finds the next outdoor observation at or after each indoor
# timestamp. This provides:
# next_outdoor_ts
# next_outdoor_temp
# next_outdoor_humidity
# and also the additional outsideMinTemp and outsideMaxTemp columns carried by the
# forward weather row.
#
# The total outdoor bracket width in minutes is computed as the difference between
# next_outdoor_ts and prev_outdoor_ts.
#
# A fractional interpolation weight is computed for each indoor timestamp by measuring
# how far that indoor timestamp lies between the previous and next outdoor timestamps.
#
# Infinite weights caused by zero-width denominators are replaced, and missing weights
# are filled with zero.
#
# Outdoor_Temperature is assigned by linear interpolation between the previous and next
# outdoor temperatures when the total outdoor bracket is less than or equal to 30 minutes.
# Otherwise it is set to missing.
#
# outsideHumidity is assigned in the same way using the same interpolation weight and
# the same 30-minute total-gap condition.
#
# outsideMinTemp and outsideMaxTemp are assigned directly from the forward weather row.
#
# The interpolated and attached outdoor variables are written back into the indoor event
# dataframe.
#
# -------------------------------------------------------------------------------
# 11. MIDNIGHT SLICING OF EVENT INTERVALS
# -------------------------------------------------------------------------------
# The daily aggregation step first converts the event timeline into a list of interval
# pieces that do not cross calendar-day boundaries.
#
# For every pair of adjacent event rows:
# the previous row supplies the starting timestamp and state values
# the current row supplies the ending timestamp
#
# If either boundary is missing, the interval is skipped.
#
# For each interval, the following state values are taken from the previous row and
# treated as the values active over that interval:
# Temperature
# Setpoint
# Outdoor_Temperature
# outsideHumidity
# FanState
#
# The interval mode is taken from curr_row["prior_mode"], which corresponds to the
# operating mode active across that interval.
#
# FanState is normalized into a binary fan_on indicator. The indicator is one if the
# lowercased string representation is one of:
# "on", "active", "true", or "1"
# Otherwise it is zero.
#
# A while loop then slices the interval into one or more pieces:
# curr_time starts at the interval start
# nxt_mid is computed as the next midnight after curr_time
# piece_end is the earlier of the true interval end and nxt_mid
# sec is the duration in seconds between curr_time and piece_end
#
# One slice record is appended for each piece with:
# Date
# sec
# mode
# temp
# setpoint
# out_temp
# out_hum
# fan_on
#
# The loop advances curr_time to piece_end and continues until the full interval
# has been partitioned.
#
# -------------------------------------------------------------------------------
# 12. DAILY GROUPING AND RUNTIME SUMMARIES
# -------------------------------------------------------------------------------
# After slicing, the list of interval pieces is converted to a dataframe.
#
# The original event dataframe also receives a Date_ext column derived from the
# event Timestamp date. This is used to group raw daily pings.
#
# For each calendar date in the slice dataframe:
# grp_slice contains the interval slices assigned to that date
# grp_ping contains the raw event rows whose timestamps fall on that same date
#
# Runtime seconds are grouped by mode and then converted to hours:
# daily_heating_hours
# daily_cooling_hours
# daily_off_hours
# daily_unknown_hours
#
# daily_runtime_hours is defined as heating plus cooling hours.
#
# total_included_known_mode_hours is defined as heating plus cooling plus off hours.
#
# total_unknown_hours duplicates the unknown-hour value.
#
# Fan runtime is computed by multiplying each slice duration by the fan_on indicator,
# summing the result, and converting seconds to hours.
#
# fan_runtime_ratio is daily_fan_on_hours divided by total_included_known_mode_hours.
# If known-mode hours are zero, the ratio is set to 0.0.
#
# -------------------------------------------------------------------------------
# 13. TIME-WEIGHTED THERMAL AND CONTROL AVERAGES
# -------------------------------------------------------------------------------
# A helper function computes weighted means for slice-based variables using slice
# duration as the weight.
#
# For each target variable, slices with non-missing values are isolated.
# If the sum of slice durations for valid values is zero, the weighted mean is missing.
# Otherwise the weighted mean is:
# sum of value times seconds divided by sum of seconds
#
# This is applied to:
# temp -> indoor_temp_time_weighted_mean
# setpoint -> setpoint_time_weighted_mean
# out_temp -> outdoor_temp_time_weighted_mean
# out_hum -> outdoor_humidity_time_weighted_mean
#
# -------------------------------------------------------------------------------
# 14. DAILY SETPOINT VOLATILITY
# -------------------------------------------------------------------------------
# setpoint_change_count is computed from the raw daily ping dataframe.
#
# The Setpoint series is filtered to non-missing values, differenced, compared against
# zero, and summed. This counts how many row-to-row setpoint changes occurred during
# that date's raw event sequence.
#
# -------------------------------------------------------------------------------
# 15. DAILY EXTREMES AND THERMODYNAMIC DELTAS
# -------------------------------------------------------------------------------
# The daily feature dictionary stores outdoor min and max values using the raw daily
# ping dataframe:
# outsideMinTemp is the daily minimum of the attached outsideMinTemp field
# outsideMaxTemp is the daily maximum of the attached outsideMaxTemp field
#
# Two delta features are then created:
# temp_gradient_mean equals indoor_temp_time_weighted_mean minus outdoor_temp_time_weighted_mean
# setpoint_gap_mean equals setpoint_time_weighted_mean minus indoor_temp_time_weighted_mean
#
# -------------------------------------------------------------------------------
# 16. CALENDAR FEATURES
# -------------------------------------------------------------------------------
# The daily date is converted into:
# day_of_week
# is_weekend
# month
#
# Circular month encoding is added using:
# month_sin
# month_cos
#
# -------------------------------------------------------------------------------
# 17. DISTRIBUTIONAL STATISTICS FOR CONTINUOUS VARIABLES
# -------------------------------------------------------------------------------
# A helper called stats_for_series computes descriptive statistics for one numeric
# series after coercing it to numeric form and dropping missing values.
#
# The output always contains a stable set of fields. If there are too few observations
# for a particular statistic, that statistic remains missing.
#
# The helper records:
# count_nonnull
# min
# 25th percentile
# median
# 75th percentile
# max
# range
# mean
# standard deviation
# variance
# interquartile range
# skewness
# excess kurtosis
# raw second moment
# raw third moment
#
# The helper is applied separately to the raw daily ping values of:
# Temperature
# Setpoint
# Outdoor_Temperature
# outsideHumidity
#
# The resulting fields are prefixed as:
# indoor_temp_
# setpoint_
# outdoor_temp_
# outdoor_hum_
#
# -------------------------------------------------------------------------------
# 18. DAILY OUTPUT CONSTRUCTION
# -------------------------------------------------------------------------------
# One feature dictionary is built for each equipment-day combination. It contains:
# equipment identifier
# date
# daily runtime accounting values
# fan metrics
# setpoint volatility count
# time-weighted averages
# daily outdoor extrema
# thermodynamic delta features
# day-of-week and seasonal fields
# the full set of descriptive statistics for indoor temperature, setpoint, outdoor
# temperature, and outdoor humidity
#
# All daily feature dictionaries are converted into a dataframe, sorted by Date,
# and reindexed.
#
# -------------------------------------------------------------------------------
# 19. MASTER LOOP OVER ALL INDOOR FILES
# -------------------------------------------------------------------------------
# The full processing function is run once for each indoor CSV file.
#
# For each indoor file, the pipeline executes:
# read raw file
# standardize columns
# parse and sort timestamps
# add Equipment_ID and raw_row_id
# resolve same-timestamp bursts
# normalize running mode
# inject virtual expiration rows
# forward-fill step-like states
# bounded-interpolate indoor temperature
# compute interval durations and per-mode seconds
# attach interpolated weather
# aggregate to daily features
#
# The resulting daily dataframe is written to the processed_daily output folder using
# the equipment ID in the filename.
#
# A console message is printed after each file is processed.
#
# -------------------------------------------------------------------------------
# CURRENT NOTE ABOUT OCCUPIED
# -------------------------------------------------------------------------------
# Occupied is listed as a step-like column and will be forward-filled if it already
# exists in the event dataframe, but the current synthetic expiration-row construction
# does not explicitly assign an Occupied reset value. In the current implementation,
# Occupied is therefore not explicitly terminated at blackout boundaries by the same
# mechanism used for RunningMode, Setpoint, and FanState.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
=========================================================================================
BLOCK 1: CORE INFRASTRUCTURE AND SYSTEM GLOBALS
=========================================================================================
This block imports necessary mathematical and filesystem libraries. It strictly defines
the global constants for the entire notebook.
=========================================================================================
"""

# import the operating system module for path manipulation
#
import os

# import the glob module to dynamically discover file paths using wildcards
#
import glob

# import the numpy library for highly optimized vectorized mathematical operations
#
import numpy as np

# import the pandas library to manage and manipulate complex tabular data structures
#
import pandas as pd

# import scikit-learn modules for feature normalization, socring and modeling
#
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

# import lightgbm to execute optimized gradient boosting ensemble logic
#
from lightgbm import LGBMRegressor

pd.set_option('future.no_silent_downcasting', True)

# define the master directory path containing the raw indoor and outdoor csv files
#
BASE_PATH = "/content/drive/MyDrive/raw_indoor_plus_outdoor/"

# define the output directory for daily aggregated statistical files
#
OUT_DAILY = os.path.join(BASE_PATH, "processed_daily")

# define the time threshold in minutes for state expiration and dead-system logic
#
TIME_THRESHOLD_MINUTES = 30

# define the numeric token used to terminate forward-filling in numerical columns
#
NUMERIC_RESET_TOKEN = -9999.9

# define step-like indoor columns that should utilize the virtual row termination architecture
#
STEP_LIKE_COLS = [
    "Setpoint", "Mode", "OutputState", "FanState"
]

# define mathematical columns that must be explicitly hidden to prevent training data leakage
#
LEAKAGE_COLS = [
    "daily_runtime_hours",
    "daily_heating_hours",
    "daily_cooling_hours",
    "daily_off_hours",
    "daily_unknown_hours",
    "total_included_known_mode_hours",
    "total_unknown_hours"
]

# ensure the output daily directory exists for file serialization
#
os.makedirs(OUT_DAILY, exist_ok=True)

# find all raw sensor files in the base path and sort them to ensure deterministic order
#
all_raw = sorted(glob.glob(os.path.join(BASE_PATH, "*.csv")))

# isolate the specific file that contains the word 'outdoor' in its name for weather processing
#
outdoor_file = "/content/drive/MyDrive/raw_indoor_plus_outdoor/outdoorweather.csv"

# treat all remaining csv files in the directory strictly as indoor thermostat telemetry sources
#
indoor_files = [f for f in all_raw if f != outdoor_file]

# output confirmation to the console to verify file discovery
#
print(f"Found 1 Outdoor File: {os.path.basename(outdoor_file)}")
print(f"Found {len(indoor_files)} Indoor Files.")

Found 1 Outdoor File: outdoorweather.csv
Found 100 Indoor Files.


In [ ]:

"""
=========================================================================================
BLOCK 2: OUTDOOR WEATHER PREPARATION
=========================================================================================
This block ingests and standardizes the globally shared outdoor weather file.
It ensures the ambient temperature and humidity data is clean, chronologically sorted,
and stripped of any duplicate timestamp anomalies.
=========================================================================================
"""

def process_outdoor_weather(path):

    # read the raw outdoor weather file using the explicit semicolon separator
    #
    dfw = pd.read_csv(path, sep=";", low_memory=False)

    # standardize the timestamp column and drop any rows with unparseable corrupt time data
    #
    dfw["Timestamp"] = pd.to_datetime(dfw["Timestamp"], errors="coerce")

    # eradicate corrupt timestamps and re-sort the timeline chronologically
    #
    dfw = dfw.dropna(subset=["Timestamp"]).sort_values("Timestamp").reset_index(drop=True)

    # map generic 'Temperature' to 'Outdoor_Temperature' to prevent indoor namespace collisions
    #
    dfw = dfw.rename(columns={"Temperature": "Outdoor_Temperature"})

    # define the explicit list of required outdoor thermodynamic variables
    #
    target_cols = ["Outdoor_Temperature", "outsideMinTemp", "outsideMaxTemp", "outsideHumidity"]

    # iterate over the target columns to enforce numeric types and coerce textual errors to NaNs
    #
    for c in target_cols:

        # forcefully convert the column data to numeric values
        #
        dfw[c] = pd.to_numeric(dfw[c], errors="coerce")

    # return the perfectly standardized and isolated weather matrix
    #
    return dfw[["Timestamp"] + target_cols]

# execute the defined function on the globally identified outdoor file
#
w_df = process_outdoor_weather(outdoor_file)

# display a pandas snippet confirming the successful creation of the weather dataframe
#
print("\nCONFIRMATION: OUTDOOR WEATHER PROCESSING")
print(w_df.head())




CONFIRMATION: OUTDOOR WEATHER PROCESSING
            Timestamp  Outdoor_Temperature  outsideMinTemp  outsideMaxTemp  \
0 2025-01-27 20:52:46                 2.94             NaN            3.10   
1 2025-01-27 21:07:47                 0.39             NaN            1.92   
2 2025-01-27 21:22:46                 0.39             NaN            1.92   
3 2025-01-27 21:37:48                 0.39             NaN            1.92   
4 2025-01-27 21:52:46                 0.29             NaN            1.88   

   outsideHumidity  
0               41  
1               52  
2               52  
3               52  
4               52  


In [ ]:
"""
=========================================================================================
BLOCK 3: INDOOR FILE INGESTION AND SCHEMA STANDARDIZATION
=========================================================================================
This block establishes the processing pipeline for a single indoor unit. It selects a
sample file, maps raw manufacturer columns to canonical names, and protects nulls.
It fails loudly via KeyError if expected schema maps are totally missing.
=========================================================================================
"""

def standardize_indoor_columns(df):
    # define the explicit mapping dictionary based strictly on observed raw schema column names
    #
    rename_map = {
        "output_state": "OutputState",
        "fan_state": "FanState",
        "running_mode": "RunningMode",
        "temp": "Temperature",
        "set_point": "Setpoint"
    }

    # apply the rename map to the dataframe to permanently enforce the canonical nomenclature
    #
    df = df.rename(columns=rename_map)

    # return the structurally updated dataframe to the caller
    #
    return df

# select the absolute first indoor file to serve as the step-by-step sample
#
sample_path = indoor_files[0]

# ingest the raw sample file utilizing the semicolon separator
#
sample_df = pd.read_csv(sample_path, sep=";", low_memory=False)

# execute the standardizer function to enforce the pipeline column names
#
sample_df = standardize_indoor_columns(sample_df)

# forcefully convert the primary timestamp column to pandas datetime objects
#
sample_df["Timestamp"] = pd.to_datetime(sample_df["Timestamp"], errors="coerce")

# radically drop any observations where the timestamp failed to parse
#
sample_df = sample_df.dropna(subset=["Timestamp"]).sort_values("Timestamp").reset_index(drop=True)

# extract and explicitly inject the equipment ID string from the file path
#
sample_df["Equipment_ID"] = os.path.splitext(os.path.basename(sample_path))[0]

# record the original physical row index to ensure mathematical stability for ties
#
sample_df["raw_row_id"] = np.arange(len(sample_df))

# display a pandas snippet confirming the successful ingestion and mapping of the sample
#
print(f"\nCONFIRMATION: RAW INGESTION FOR {sample_df['Equipment_ID'].iloc[0]}")
print(sample_df[["Timestamp", "Equipment_ID", "RunningMode", "Temperature", "Setpoint"]].head())


CONFIRMATION: RAW INGESTION FOR timeseries_table_timeseries_table (1)
            Timestamp                           Equipment_ID RunningMode  \
0 2025-02-05 15:53:08  timeseries_table_timeseries_table (1)         NaN   
1 2025-02-05 15:53:10  timeseries_table_timeseries_table (1)         off   
2 2025-02-05 16:05:10  timeseries_table_timeseries_table (1)         NaN   
3 2025-02-05 16:39:33  timeseries_table_timeseries_table (1)         NaN   
4 2025-02-05 16:39:34  timeseries_table_timeseries_table (1)         off   

   Temperature  Setpoint  
0          NaN       NaN  
1          NaN       NaN  
2          NaN      23.0  
3          NaN      23.0  
4          NaN       NaN  


In [ ]:

"""
=========================================================================================
BLOCK 4: BURST RESOLUTION (INTRA-SECOND SNAPSHOT SYNTHESIS)
=========================================================================================
This block resolves instances where the IoT device sent multiple updates in the exact
same second. It mathematically forward-fills within the burst and extracts the final
row, creating a perfect snapshot of all sensor states at that micro-moment.
=========================================================================================
"""

def resolve_same_timestamp_bursts(df):

    # isolate all data columns, intentionally excluding the timestamp for localized resolution
    #
    data_cols = [c for c in df.columns if c != "Timestamp"]

    # group the dataframe strictly by timestamp and perform a downward fill within the burst
    #
    df[data_cols] = df.groupby("Timestamp", sort=False)[data_cols].ffill()

    # extract only the final row from each timestamp group to capture the fully resolved state
    #
    last_rows = df.groupby("Timestamp", sort=False, as_index=False).tail(1)

    # reset indices to guarantee sequential chronological access downstream
    #
    return last_rows.reset_index(drop=True)

# execute the burst resolution algorithm on the active sample dataframe
#
sample_df = resolve_same_timestamp_bursts(sample_df)

# display a pandas snippet confirming the burst resolution eliminated duplicate timestamps
#
print("\nCONFIRMATION: BURST RESOLUTION")
print(sample_df[["Timestamp", "raw_row_id", "RunningMode", "Temperature"]].head())




CONFIRMATION: BURST RESOLUTION
            Timestamp  raw_row_id RunningMode  Temperature
0 2025-02-05 15:53:08           0         NaN          NaN
1 2025-02-05 15:53:10           1         off          NaN
2 2025-02-05 16:05:10           2         NaN          NaN
3 2025-02-05 16:39:33           3         NaN          NaN
4 2025-02-05 16:39:34           4         off          NaN


In [ ]:
"""
=========================================================================================
BLOCK 5: STATE NORMALIZATION AND VIRTUAL EXPIRATION ROW INJECTION
=========================================================================================
This block first maps the raw mode strings into canonical states. It then calculates
the time gap between consecutive pings. If the gap strictly exceeds 30 minutes, it
injects a synthetic row exactly 30 minutes into the blackout. This row acts as a brick
wall, forcefully terminating system trust by resetting values.
=========================================================================================
"""

def normalize_running_mode(x):
    # check for mathematical nulls and return np.nan to explicitly protect the blank row
    #
    if pd.isna(x):
        return np.nan

    # normalize the raw text input by converting to lowercase and stripping invisible whitespace
    #
    s = str(x).strip().lower()

    # categorize the normalized string cleanly into the heating operational bucket
    #
    if s in {"heat", "heating"}:
        return "heat"

    # categorize the normalized string cleanly into the cooling operational bucket
    #
    if s in {"cool", "cooling"}:
        return "cool"

    # categorize the normalized string cleanly into the idle operational bucket
    #
    if s in {"off", "idle", "none", "false", "0"}:
        return "off"

    # default all unparseable or completely missing text representations to the unknown bucket
    #
    return "unknown"

def inject_virtual_expiration_rows(df):
    # calculate the difference between consecutive chronological timestamps in absolute seconds
    #
    df["gap_sec"] = df["Timestamp"].diff().dt.total_seconds()

    # explicitly isolate the rows that follow a gap greater than the 30-minute pipeline threshold
    #
    blackouts = df[df["gap_sec"] > (TIME_THRESHOLD_MINUTES * 60)].copy()

    # initialize an empty list to dynamically collect the synthesized virtual expiration rows
    #
    v_rows = []

    # iterate line-by-line through detected blackout boundaries to generate explicit expiration markers
    #
    for idx, row in blackouts.iterrows():

        # define a new dictionary representation for the synthetic marker row
        #
        v = {}

        # calculate the exact start of the gap by subtracting the gap duration from the current time
        #
        gap_start = row["Timestamp"] - pd.Timedelta(seconds=row["gap_sec"])

        # set the synthetic timestamp to exactly 30 minutes after the last confirmed system ping
        #
        v["Timestamp"] = gap_start + pd.Timedelta(minutes=TIME_THRESHOLD_MINUTES)

        # copy the equipment identifier to seamlessly preserve unit-level integrity
        #
        v["Equipment_ID"] = row["Equipment_ID"]

        # forcefully reset the operational mode to terminate forward-filling persistence
        #
        v["RunningMode_clean"] = "unknown"

        # forcefully apply the mathematical reset token to terminate numeric state persistence
        #
        v["Setpoint"] = NUMERIC_RESET_TOKEN

        # forcefully reset the string-based fan state to an unknown designation
        #
        v["FanState"] = "unknown"

        # forcefully terminate the mechanical and user modes to prevent hardware hallucination
        #
        v["OutputState"] = "unknown"
        v["Mode"] = "unknown"

        # flag this row as an explicitly synthetic injection to maintain perfect pipeline transparency
        #
        v["is_virtual_expiration"] = True

        # append the constructed virtual row representation to the temporary tracking list
        #
        v_rows.append(v)

    # mathematically concatenate the synthetic rows into the main timeline if any were actually generated
    #
    if v_rows:

        # join the original timeline matrix with the new synthetic matrix
        #
        df = pd.concat([df, pd.DataFrame(v_rows)], ignore_index=True)

    # violently re-sort the entire integrated timeline to position virtual markers flawlessly within their gaps
    #
    df = df.sort_values("Timestamp").reset_index(drop=True)

    # remove the temporary mathematical gap column to cleanly maintain schema purity
    #
    return df.drop(columns=["gap_sec"])

# safely extract the raw running mode into an audit tracking column using a dictionary get-method
#
sample_df["RunningMode_raw"] = sample_df.get("RunningMode", np.nan)

# map the raw mode strings into the canonical representations while strictly protecting NaNs
#
sample_df["RunningMode_clean"] = sample_df["RunningMode_raw"].map(normalize_running_mode)

# execute the blackout injection logic on the active sample dataframe
#
sample_df = inject_virtual_expiration_rows(sample_df)

# display a pandas snippet explicitly isolating and confirming the newly minted virtual rows
#
print("\nCONFIRMATION: VIRTUAL EXPIRATION ROWS INJECTED")
print(sample_df[sample_df.get('is_virtual_expiration') == True][["Timestamp", "RunningMode_clean", "Setpoint", "OutputState"]].head())


CONFIRMATION: VIRTUAL EXPIRATION ROWS INJECTED
             Timestamp RunningMode_clean  Setpoint OutputState
3  2025-02-05 16:35:10           unknown   -9999.9     unknown
6  2025-02-05 17:09:34           unknown   -9999.9     unknown
8  2025-02-05 18:09:25           unknown   -9999.9     unknown
10 2025-02-05 19:09:17           unknown   -9999.9     unknown
12 2025-02-05 20:09:09           unknown   -9999.9     unknown


In [ ]:
"""
=========================================================================================
BLOCK 6: UNBOUNDED STATE PERSISTENCE & DEAD-SYSTEM INTERPOLATION
=========================================================================================
This block triggers an unbounded forward-fill. Because the Virtual Rows exist, the
fill mathematically stops at 30 minutes. It also executes a dead-system interpolation
logic that strictly evaluates the TOTAL gap width.
=========================================================================================
"""

def bounded_time_interpolate(series, times):
    # forcefully cast the target series to numeric floating points and protect pure NaNs
    #
    s = pd.to_numeric(series, errors="coerce")

    # isolate the exact chronological timestamp of the previous valid point before the missing bracket
    #
    prev_valid_ts = times.where(s.notna()).ffill()

    # isolate the exact chronological timestamp of the next valid point following the missing bracket
    #
    next_valid_ts = times.where(s.notna()).bfill()

    # calculate the total comprehensive chronological gap between the boundary timestamps in minutes
    #
    total_gap_min = (next_valid_ts - prev_valid_ts).dt.total_seconds() / 60.0

    # execute standard mathematical time-based interpolation utilizing a temporary timestamp-indexed dataframe
    #
    temp = pd.DataFrame({"t": times, "v": s}).set_index("t")

    # extract the mathematically interpolated continuous values as a raw numpy array
    #
    interp_vals = temp["v"].interpolate(method="time").values

    # establish a boolean mask ensuring the total bracket strictly falls under the 30-minute tolerance
    #
    ok_mask = s.notna() | (total_gap_min <= TIME_THRESHOLD_MINUTES)

    # apply the logic gate utilizing a numpy where-clause to return NaNs if the system was mathematically dead
    #
    return pd.Series(np.where(ok_mask, interp_vals, np.nan), index=series.index)

# execute an unbounded downward fill; the virtual expiration rows will organically act as brick walls
#
sample_df["RunningMode"] = sample_df["RunningMode_clean"].ffill()

# execute dead-system continuous interpolation exclusively on the indoor ambient temperature readings
#
sample_df["Temperature"] = bounded_time_interpolate(sample_df["Temperature"], sample_df["Timestamp"])

# static metadata persists backwards and forwards across the entire house timeline ignoring blackouts
#
if "RentalStatus" in sample_df.columns:
    sample_df["RentalStatus"] = sample_df["RentalStatus"].ffill().bfill()

# iterate sequentially through every explicit step-like administrative field to apply unbounded persistence logic
#
for c in STEP_LIKE_COLS:
    if c in sample_df.columns:
        # forcefully coerce to numeric directly before forward-filling to securely protect numeric token logic
        #
        if c == "Setpoint":
            sample_df[c] = pd.to_numeric(sample_df[c], errors="coerce")

        # perform the unbounded downward fill operation across the entire timeline sequence
        #
        sample_df[c] = sample_df[c].ffill()

        # safely replace the synthetic numeric reset tokens with pure NaNs to terminate state at the virtual boundary
        #
        if c == "Setpoint":
            sample_df[c] = sample_df[c].replace(NUMERIC_RESET_TOKEN, np.nan)

# re-sort the final timeline to ensure all synthetic markers and filled values align chronologically perfectly
#
sample_df = sample_df.sort_values(["Timestamp", "raw_row_id"], kind="mergesort").reset_index(drop=True)

# display a pandas snippet explicitly confirming the state persistence logic operated effectively
#
print("\nCONFIRMATION: UNBOUNDED FILL AND INTERPOLATION")
print(sample_df[["Timestamp", "RunningMode", "Temperature", "Setpoint", "RentalStatus"]].head(10))


CONFIRMATION: UNBOUNDED FILL AND INTERPOLATION
            Timestamp RunningMode  Temperature  Setpoint RentalStatus
0 2025-02-05 15:53:08         NaN          NaN       NaN         Sold
1 2025-02-05 15:53:10         off          NaN       NaN         Sold
2 2025-02-05 16:05:10         off          NaN      23.0         Sold
3 2025-02-05 16:35:10     unknown          NaN       NaN         Sold
4 2025-02-05 16:39:33     unknown          NaN      23.0         Sold
5 2025-02-05 16:39:34         off          NaN      23.0         Sold
6 2025-02-05 17:09:34     unknown          NaN       NaN         Sold
7 2025-02-05 17:39:25         off          NaN      23.0         Sold
8 2025-02-05 18:09:25     unknown          NaN       NaN         Sold
9 2025-02-05 18:39:17         off          NaN      23.0         Sold


In [ ]:
"""
=========================================================================================
BLOCK 7: INTERVAL RUNTIME ACCOUNTING
=========================================================================================
This block calculates the exact number of seconds spanning each operational interval
and categorically assigns those seconds to the prior state.
=========================================================================================
"""

# calculate the timestamp of the immediately preceding event to define the interval initiation
#
sample_df["prev_timestamp"] = sample_df["Timestamp"].shift(1)

# algebraically compute the total duration of the interval in raw seconds and fill nulls with zero
#
sample_df["Duration_Seconds"] = (sample_df["Timestamp"] - sample_df["prev_timestamp"]).dt.total_seconds().fillna(0)

# retrieve the operational mode that was active leading strictly into the current measurement interval
#
sample_df["Interval_RunningMode"] = sample_df["RunningMode"].shift(1).fillna("unknown")

# establish vectorized duration tracking
#
event_dt = sample_df["Duration_Seconds"]

# execute heavily vectorized runtime accounting to credit heating intervals based strictly on prior mode
#
sample_df["heat_sec"] = np.where(sample_df["Interval_RunningMode"] == "heat", event_dt, 0.0)

# execute heavily vectorized runtime accounting to credit cooling intervals based strictly on prior mode
#
sample_df["cool_sec"] = np.where(sample_df["Interval_RunningMode"] == "cool", event_dt, 0.0)

# execute heavily vectorized runtime accounting to credit inactive intervals based strictly on prior mode
#
sample_df["off_sec"] = np.where(sample_df["Interval_RunningMode"] == "off", event_dt, 0.0)

# execute heavily vectorized runtime accounting to credit unlogged intervals based strictly on prior mode
#
sample_df["unk_sec"] = np.where(sample_df["Interval_RunningMode"] == "unknown", event_dt, 0.0)

# display a pandas snippet explicitly confirming interval durations were successfully accounted for
#
print("\nCONFIRMATION: INTERVAL ACCOUNTING IN SECONDS")
print(sample_df[["Timestamp", "Duration_Seconds", "Interval_RunningMode", "heat_sec", "cool_sec", "unk_sec"]].head(10))


CONFIRMATION: INTERVAL ACCOUNTING IN SECONDS
            Timestamp  Duration_Seconds Interval_RunningMode  heat_sec  \
0 2025-02-05 15:53:08               0.0              unknown       0.0   
1 2025-02-05 15:53:10               2.0              unknown       0.0   
2 2025-02-05 16:05:10             720.0                  off       0.0   
3 2025-02-05 16:35:10            1800.0                  off       0.0   
4 2025-02-05 16:39:33             263.0              unknown       0.0   
5 2025-02-05 16:39:34               1.0              unknown       0.0   
6 2025-02-05 17:09:34            1800.0                  off       0.0   
7 2025-02-05 17:39:25            1791.0              unknown       0.0   
8 2025-02-05 18:09:25            1800.0                  off       0.0   
9 2025-02-05 18:39:17            1792.0              unknown       0.0   

   cool_sec  unk_sec  
0       0.0      0.0  
1       0.0      2.0  
2       0.0      0.0  
3       0.0      0.0  
4       0.0    263.0  
5

In [ ]:

"""
=========================================================================================
BLOCK 8: SPATIOTEMPORAL WEATHER ALIGNMENT
=========================================================================================
This block executes merge_asof to align the outdoor timeline. It evaluates the true
width of the outdoor gap. If the gap is under 30 minutes, it interpolates the
point variables mathematically.
=========================================================================================
"""

def attach_weather_to_indoor(indoor_df, outdoor_df):
    # mathematically isolate strictly the unique indoor timestamps for targeted temporal matching
    #
    indoor_ts = indoor_df[["Timestamp"]].copy().sort_values("Timestamp")

    # execute a backwards merge_asof to discover the absolute previous valid outdoor reading
    #
    prev_w = pd.merge_asof(
        indoor_ts,
        outdoor_df[["Timestamp", "Outdoor_Temperature", "outsideHumidity"]].rename(columns={
            "Timestamp": "prev_outdoor_ts",
            "Outdoor_Temperature": "prev_outdoor_temp",
            "outsideHumidity": "prev_outdoor_humidity"
        }),
        left_on="Timestamp", right_on="prev_outdoor_ts", direction="backward"
    )

    # execute a forwards merge_asof to discover the absolute next valid outdoor reading
    #
    next_w = pd.merge_asof(
        indoor_ts,
        outdoor_df.rename(columns={
            "Timestamp": "next_outdoor_ts",
            "Outdoor_Temperature": "next_outdoor_temp",
            "outsideHumidity": "next_outdoor_humidity"
        }),
        left_on="Timestamp", right_on="next_outdoor_ts", direction="forward"
    )

    # calculate the total true comprehensive gap between absolute outdoor readings
    #
    total_gap_min = (next_w["next_outdoor_ts"] - prev_w["prev_outdoor_ts"]).dt.total_seconds() / 60.0

    # construct a rigorous boolean mask ensuring the outdoor data bracket satisfies the 30-minute tolerance
    #
    ok_mask = total_gap_min <= TIME_THRESHOLD_MINUTES

    # calculate the fractional interpolation progress of the indoor timestamp traversing the bracket
    #
    weight = (indoor_ts["Timestamp"] - prev_w["prev_outdoor_ts"]).dt.total_seconds() / (total_gap_min * 60.0)

    # gracefully handle exact mathematical timestamp matches by explicitly replacing infinities with nulls
    #
    weight = weight.replace([np.inf, -np.inf], np.nan).fillna(0)

    # mathematically apply the linear interpolation formula and the 30-minute blackout logic gate mask
    #
    ext_temp = np.where(ok_mask, prev_w["prev_outdoor_temp"] * (1 - weight) + next_w["next_outdoor_temp"] * weight, np.nan)

    # mathematically apply the linear interpolation formula for humidity utilizing the identical progression weights
    #
    ext_hum = np.where(ok_mask, prev_w["prev_outdoor_humidity"] * (1 - weight) + next_w["next_outdoor_humidity"] * weight, np.nan)

    # directly attach the fully interpolated ambient variables back to the primary indoor dataframe
    #
    indoor_df["Outdoor_Temperature"] = ext_temp
    indoor_df["outsideHumidity"] = ext_hum

    # strictly backfill formal interval summaries sourced directly from the closing outdoor sequence observation
    #
    indoor_df["outsideMinTemp"] = next_w["outsideMinTemp"]
    indoor_df["outsideMaxTemp"] = next_w["outsideMaxTemp"]

    # return the perfectly integrated thermodynamic matrix
    #
    return indoor_df

# execute the weather attachment pipeline on the active sample dataframe
#
sample_df = attach_weather_to_indoor(sample_df, w_df)

# display a pandas snippet confirming the successful alignment of outdoor variables
#
print("\nCONFIRMATION: OUTDOOR WEATHER ALIGNMENT")
print(sample_df[["Timestamp", "Outdoor_Temperature", "outsideHumidity", "outsideMinTemp"]].head(10))



CONFIRMATION: OUTDOOR WEATHER ALIGNMENT
            Timestamp  Outdoor_Temperature  outsideHumidity  outsideMinTemp
0 2025-02-05 15:53:08            10.692442        88.073252           10.53
1 2025-02-05 15:53:10            10.692664        88.079911           10.53
2 2025-02-05 16:05:10            10.772575        90.477248           10.53
3 2025-02-05 16:35:10            10.659467        91.000000            9.97
4 2025-02-05 16:39:33            10.627622        91.118889            9.97
5 2025-02-05 16:39:34            10.627600        91.120000            9.97
6 2025-02-05 17:09:34            10.632614        92.000000           10.53
7 2025-02-05 17:39:25            10.818700        92.220000           10.69
8 2025-02-05 18:09:25            11.575200        95.220000           10.70
9 2025-02-05 18:39:17            11.862022        97.000000           10.70


In [ ]:
"""
=========================================================================================
BLOCK 9: MIDNIGHT SLICING, STATISTICAL ROLL-UPS, AND ADVANCED FEATURE ENGINEERING
=========================================================================================
This block distributes operational seconds across calendar days using a recursive midnight slicer.
It computes time-weighted thermodynamic averages, calculates 15-moment distributional statistics,
and engineers seasonality and thermodynamic deltas using strictly canonical column names.
=========================================================================================
"""

def stats_for_series(s, prefix):
    """
    function: stats_for_series
    arguments: s (pandas Series), prefix (string for column names)
    return: dictionary of 15 robust statistical moments
    description: Computes distribution statistics safely without try-except blocks.
    It evaluates sample size before executing variance/skewness to prevent math errors.
    """
    # forcefully coerce the input series to numeric floating points and drop nulls
    #
    s = pd.to_numeric(s, errors="coerce").dropna()

    # calculate the absolute integer count of valid observations available for math
    #
    n = len(s)

    # initialize the output dictionary explicitly recording the valid observation count
    #
    out = {f"{prefix}_count_nonnull": n}

    # define the explicit list of statistical keys required for the feature matrix
    #
    keys = [
        "min", "q25", "median", "q75", "max", "range", "mean", "std",
        "variance", "iqr", "skewness", "kurtosis_excess", "raw_moment_2", "raw_moment_3"
    ]

    # pre-fill all statistical keys with nan to guarantee stable schema generation
    #
    for k in keys:
        out[f"{prefix}_{k}"] = np.nan

    # if the series is entirely empty, safely return the nan-filled dictionary immediately
    #
    if n == 0:
        return out

    # calculate the 25th percentile boundary for the interquartile range
    #
    q25 = s.quantile(0.25)

    # calculate the 75th percentile boundary for the interquartile range
    #
    q75 = s.quantile(0.75)

    # update the dictionary with base metrics that require at least 1 observation
    #
    out.update({
        f"{prefix}_min": s.min(),
        f"{prefix}_q25": q25,
        f"{prefix}_median": s.median(),
        f"{prefix}_q75": q75,
        f"{prefix}_max": s.max(),
        f"{prefix}_range": s.max() - s.min(),
        f"{prefix}_mean": s.mean(),
        f"{prefix}_iqr": q75 - q25,
        f"{prefix}_raw_moment_2": np.mean(np.power(s, 2)),
        f"{prefix}_raw_moment_3": np.mean(np.power(s, 3)),
    })

    # standard deviation and variance strictly require at least 2 observations
    #
    if n >= 2:
        out[f"{prefix}_std"] = s.std(ddof=1)
        out[f"{prefix}_variance"] = s.var(ddof=1)

    # mathematical skewness strictly requires at least 3 observations to evaluate tail geometry
    #
    if n >= 3:
        out[f"{prefix}_skewness"] = s.skew()

    # mathematical kurtosis strictly requires at least 4 observations to evaluate peak shape
    #
    if n >= 4:
        out[f"{prefix}_kurtosis_excess"] = s.kurt()

    # return the fully populated statistical dictionary
    #
    return out


def aggregate_to_daily(df_event):
    """
    function: aggregate_to_daily
    arguments: df_event (cleaned, event-level timeline)
    return: daily_df (rich, 100+ feature aggregated matrix)
    description: Slices intervals, computes weighted means, extracts moments, and builds deltas.
    """
    # initialize an array to hold the mathematically sliced temporal pieces
    #
    slices = []

    # iteratively evaluate every physical interval spanning the event dataframe
    #
    for i in range(1, len(df_event)):

        # systematically identify the previous row to extract starting states
        #
        prev_row = df_event.iloc[i-1]

        # identify the current row to define the termination boundary
        #
        curr_row = df_event.iloc[i]

        # establish the chronological boundaries of the active interval
        #
        start = prev_row["Timestamp"]
        end = curr_row["Timestamp"]

        # aggressively bypass the slicing if temporal bounds are missing
        #
        if pd.isna(start) or pd.isna(end):
            continue

        # extract the driving operational and thermodynamic states using canonical pipeline names
        #
        running_mode = curr_row["Interval_RunningMode"]
        temperature = prev_row.get("Temperature", np.nan)
        setpoint = prev_row.get("Setpoint", np.nan)

        # cleanly extract the true mechanical hardware state string for interval tracking
        #
        output_state = str(prev_row.get("OutputState", "unknown")).strip().lower()

        # safely extract raw textual fan representations from the previous interval
        #
        fan_raw = str(prev_row.get("FanState", "unknown")).strip().lower()

        # determine if the fan was actively circulating air during this interval
        #
        if fan_raw in {"on", "active", "true", "1"}:
            fan_state_active = 1
        else:
            fan_state_active = 0

        # establish the recursive tracking cursor strictly at the interval initiation
        #
        curr_time = start

        # initiate a continuous while-loop to recursively slice the interval chronologically
        #
        while curr_time < end:

            # computationally calculate the exact datetime object representing absolute midnight
            #
            nxt_mid = curr_time.normalize() + pd.Timedelta(days=1)

            # rigorously constrain the temporal termination to whichever boundary occurs earliest
            #
            piece_end = min(end, nxt_mid)

            # algebraically calculate the raw physical duration spanning this architectural piece
            #
            sec = (piece_end - curr_time).total_seconds()

            # cleanly append the robust, date-constrained mathematical piece strictly preserving canonical names
            #
            slices.append({
                "Date": curr_time.date(),
                "Duration_Seconds": sec,
                "RunningMode": running_mode,
                "Temperature": temperature,
                "Setpoint": setpoint,
                "FanState_Active": fan_state_active,
                "OutputState": output_state
            })

            # explicitly advance the tracking cursor forward to seamlessly match the termination bound
            #
            curr_time = piece_end

    # securely convert the populated array of pieces into a highly optimized pandas dataframe
    #
    res = pd.DataFrame(slices)

    # explicitly define a pure date column on the raw event dataframe for ping-based grouping
    #
    df_event["Date_ext"] = df_event["Timestamp"].dt.date

    # initialize an empty list to store the final compiled daily feature dictionaries
    #
    daily_rows = []

    # iterate explicitly through every unique discrete calendar date mathematically observed
    #
    for date, grp_slice in res.groupby("Date"):

        # isolate the raw ping events that occurred strictly on this specific calendar date
        #
        grp_ping = df_event[df_event["Date_ext"] == date]

        # algebraically calculate the aggregate sum of seconds strictly grouped by canonical mode
        #
        m_sec = grp_slice.groupby("RunningMode")["Duration_Seconds"].sum().to_dict()

        # rigorously extract the total fractional hours, defaulting cleanly to zero
        #
        h_hrs = m_sec.get("heat", 0) / 3600.0
        c_hrs = m_sec.get("cool", 0) / 3600.0
        o_hrs = m_sec.get("off", 0) / 3600.0
        u_hrs = m_sec.get("unknown", 0) / 3600.0

        # mathematically define total known runtime hours as the primary predictive target
        #
        rt_hrs = h_hrs + c_hrs

        # explicitly evaluate physical tracking viability to safely poison hallucinated targets
        #
        if u_hrs > 3.0:
            rt_hrs = np.nan

        # define total known hours explicitly included in the daily sequence
        #
        known_hrs = h_hrs + c_hrs + o_hrs

        # calculate the absolute duration the fan was active in fractional hours
        #
        fan_sec_total = (grp_slice["FanState_Active"] * grp_slice["Duration_Seconds"]).sum()
        fan_hrs = fan_sec_total / 3600.0

        # mathematically define the runtime ratio strictly against known tracking hours to prevent division by zero
        #
        if known_hrs > 0:
            fan_ratio = fan_hrs / known_hrs
        else:
            fan_ratio = 0.0

        # calculate highly granular mechanical hardware state durations using canonical groupings
        #
        os_sec = grp_slice.groupby("OutputState")["Duration_Seconds"].sum().to_dict()
        out_idle_hrs = os_sec.get("idle", 0) / 3600.0
        out_heat_high_hrs = os_sec.get("heat_high", 0) / 3600.0
        out_heat_med_hrs = os_sec.get("heat_med", 0) / 3600.0
        out_heat_low_hrs = os_sec.get("heat_low", 0) / 3600.0
        out_pre_heat_hrs = os_sec.get("pre_heat", 0) / 3600.0
        out_post_heat_hrs = os_sec.get("post_heat", 0) / 3600.0
        out_refresh_heat_hrs = os_sec.get("refresh_heat", 0) / 3600.0
        out_cool_high_hrs = os_sec.get("cool_high", 0) / 3600.0
        out_cool_med_hrs = os_sec.get("cool_med", 0) / 3600.0
        out_cool_low_hrs = os_sec.get("cool_low", 0) / 3600.0
        out_pre_cool_hrs = os_sec.get("pre_cool", 0) / 3600.0
        out_post_cool_hrs = os_sec.get("post_cool", 0) / 3600.0
        out_refresh_cool_hrs = os_sec.get("refresh_cool", 0) / 3600.0

        # define a local helper to safely compute time-weighted thermodynamic averages
        #
        def calc_weighted_mean(val_col):
            # isolate slices where the target value is not a mathematical null
            #
            valid = grp_slice[grp_slice[val_col].notna()]

            # return nan immediately if no valid data exists to prevent zero-division explosions
            #
            if valid["Duration_Seconds"].sum() == 0:
                return np.nan

            # calculate the pure weighted average based on physical state duration
            #
            return (valid[val_col] * valid["Duration_Seconds"]).sum() / valid["Duration_Seconds"].sum()

        # extract time-weighted means strictly for the interior environment variables
        #
        wt_temp = calc_weighted_mean("Temperature")
        wt_sp = calc_weighted_mean("Setpoint")

        # physically count how many times the setpoint changed to quantify user volatility
        #
        if "Setpoint" in grp_ping:
            # mathematically drop the initial diff nan to prevent counting the first daily ping as a phantom change
            #
            sp_changes = grp_ping["Setpoint"].dropna().diff().dropna().ne(0).sum()
        else:
            sp_changes = 0

        # count explicit occupancy sensor pings safely handling boolean and text variants
        #
        if "Occupied" in grp_ping:
            occ_clean = grp_ping["Occupied"].astype(str).str.strip().str.lower()
            occ_true_count = (occ_clean == "true").sum()
            occ_false_count = (occ_clean == "false").sum()
        else:
            occ_true_count = 0
            occ_false_count = 0

        # initialize the master dictionary payload for this specific calendar day
        #
        day_dict = {
            "Equipment_ID": df_event.iloc[0]["Equipment_ID"],
            "Date": pd.to_datetime(date),
            "daily_heating_hours": h_hrs,
            "daily_cooling_hours": c_hrs,
            "daily_off_hours": o_hrs,
            "daily_unknown_hours": u_hrs,
            "daily_runtime_hours": rt_hrs,
            "total_included_known_mode_hours": known_hrs,
            "total_unknown_hours": u_hrs,
            "daily_fan_on_hours": fan_hrs,
            "fan_runtime_ratio": fan_ratio,
            "setpoint_change_count": sp_changes,
            "occupied_ping_count": occ_true_count,
            "unoccupied_ping_count": occ_false_count,
            "daily_idle_hours": out_idle_hrs,
            "daily_heat_high_hours": out_heat_high_hrs,
            "daily_heat_med_hours": out_heat_med_hrs,
            "daily_heat_low_hours": out_heat_low_hrs,
            "daily_pre_heat_hours": out_pre_heat_hrs,
            "daily_post_heat_hours": out_post_heat_hrs,
            "daily_refresh_heat_hours": out_refresh_heat_hrs,
            "daily_cool_high_hours": out_cool_high_hrs,
            "daily_cool_med_hours": out_cool_med_hrs,
            "daily_cool_low_hours": out_cool_low_hrs,
            "daily_pre_cool_hours": out_pre_cool_hrs,
            "daily_post_cool_hours": out_post_cool_hrs,
            "daily_refresh_cool_hours": out_refresh_cool_hrs,
            "indoor_temp_time_weighted_mean": wt_temp,
            "setpoint_time_weighted_mean": wt_sp,
        }

        # engineer the operational comfort gap between the user target and actual reality
        #
        if pd.notna(wt_sp) and pd.notna(wt_temp):
            day_dict["setpoint_gap_mean"] = wt_sp - wt_temp
        else:
            day_dict["setpoint_gap_mean"] = np.nan

        # extract the exact day of the week to inform temporal sequences
        #
        dow = day_dict["Date"].dayofweek
        day_dict["day_of_week"] = dow

        # define a binary flag to highly weight weekend occupancy behaviors
        #
        if dow >= 5:
            day_dict["is_weekend"] = 1
        else:
            day_dict["is_weekend"] = 0

        # extract the month to evaluate general seasonal base loads
        #
        mth = day_dict["Date"].month
        day_dict["month"] = mth

        # execute circular encoding on the month to inform models that december and january are adjacent
        #
        day_dict["month_sin"] = np.sin(2 * np.pi * mth / 12)
        day_dict["month_cos"] = np.cos(2 * np.pi * mth / 12)

        # execute the robust statistical helper strictly across interior variables
        #
        day_dict.update(stats_for_series(grp_ping["Temperature"], "indoor_temp"))
        day_dict.update(stats_for_series(grp_ping["Setpoint"], "setpoint"))

        # seamlessly append the engineered dictionary into the aggregate tracking list
        #
        daily_rows.append(day_dict)

    # securely convert the fully synthesized sequence into the formalized daily dataframe and sort chronologically
    #
    return pd.DataFrame(daily_rows).sort_values("Date").reset_index(drop=True)

In [ ]:
"""
=========================================================================================
BLOCK 10: MASTER PIPELINE EXECUTION LOOP
=========================================================================================
This block compiles all previously defined steps into a master unit-processor. It
iterates through EVERY indoor file, rigorously applying the full start-to-finish logic,
and serializing the completely processed daily tables to disk.
=========================================================================================
"""

# generate the absolute true physical weather calendar decoupled from telemetry pings
#
w_df["Date"] = pd.to_datetime(w_df["Timestamp"].dt.date)

# mathematically aggregate true weather bounds safely outside the indoor hardware loop
#
true_weather = w_df.groupby("Date").agg(
    true_outside_min=("outsideMinTemp", "min"),
    true_outside_max=("outsideMaxTemp", "max"),
    true_outside_mean=("Outdoor_Temperature", "mean"),
    true_humidity_mean=("outsideHumidity", "mean")
).reset_index()

def process_one_indoor_file(path, outdoor_df, true_weather_df):

    # comprehensively orchestrate the ingestion and canonical mapping processes
    #
    raw = pd.read_csv(path, sep=";", low_memory=False)
    raw = standardize_indoor_columns(raw)

    # enforce timestamp structures and remove unparseable errors flawlessly
    #
    raw["Timestamp"] = pd.to_datetime(raw["Timestamp"], errors="coerce")
    raw = raw.dropna(subset=["Timestamp"]).sort_values("Timestamp").reset_index(drop=True)

    # securely inject the unit identifier extracted directly from the filesystem architecture
    #
    raw["Equipment_ID"] = os.path.splitext(os.path.basename(path))[0]
    raw["raw_row_id"] = np.arange(len(raw))

    # structurally resolve any observed intra-second telemetry collisions seamlessly
    #
    event = resolve_same_timestamp_bursts(raw)

    # safely extract the raw running mode into an audit tracking column using a dictionary get-method
    #
    event["RunningMode_raw"] = event.get("RunningMode", np.nan)

    # successfully map the raw operating state into the required canonical definitions
    #
    event["RunningMode_clean"] = event["RunningMode_raw"].map(normalize_running_mode)

    # completely analyze the temporal architecture to flawlessly inject blackout terminator markers
    #
    event = inject_virtual_expiration_rows(event)

    # organically apply unbounded persistence filling against the explicit blackout markers
    #
    event["RunningMode"] = event["RunningMode_clean"].ffill()

    # strictly apply the total-gap dead system bounds to explicitly interpolate missing temperatures
    #
    event["Temperature"] = bounded_time_interpolate(event["Temperature"], event["Timestamp"])

    # static metadata persists backwards and forwards across the entire house timeline ignoring blackouts
    #
    if "RentalStatus" in event.columns:
        event["RentalStatus"] = event["RentalStatus"].ffill().bfill()

    # explicitly iterate across administrative values to finalize state tracking structures
    #
    for c in STEP_LIKE_COLS:
        if c in event.columns:

            # strictly enforce typing constraints directly preceding logical propagation fills
            #
            if c == "Setpoint":
                event[c] = pd.to_numeric(event[c], errors="coerce")

            # systematically trigger the forward fill sequence
            #
            event[c] = event[c].ffill()

            # violently replace explicit logic markers with structural nulls to terminate operation cleanly
            #
            if c == "Setpoint":
                event[c] = event[c].replace(NUMERIC_RESET_TOKEN, np.nan)

    # logically define the absolute prior state to feed the recursive midnight slicer
    #
    event = event.sort_values(["Timestamp", "raw_row_id"], kind="mergesort").reset_index(drop=True)
    event["Interval_RunningMode"] = event["RunningMode"].shift(1).fillna("unknown")

    # formally convert the complex event interval manifold perfectly into the normalized daily row layout
    #
    daily = aggregate_to_daily(event)

    # strictly force the date column to pandas datetime to ensure safe physical merging
    #
    daily["Date"] = pd.to_datetime(daily["Date"])

    # left-merge the true decoupled global weather timeline onto the aggregated sequence
    #
    daily = pd.merge(daily, true_weather_df, on="Date", how="left")

    # mathematically engineer global thermodynamic gradients safe from telemetry limits
    #
    daily["temp_gradient_mean"] = daily["indoor_temp_time_weighted_mean"] - daily["true_outside_mean"]

    # return the perfectly integrated thermodynamic matrix
    #
    return daily


# execute an unbounded procedural iteration evaluating every fully available indoor telemetry stream
#
print("\nINITIATING MASSIVE PIPELINE EXECUTION FOR ALL FILES")
for f in indoor_files:

    # securely define the operational id cleanly derived exclusively from the internal filename
    #
    uid = os.path.splitext(os.path.basename(f))[0]

    # strictly execute the monolithic per-unit function logic to complete structural processing
    #
    daily = process_one_indoor_file(f, w_df, true_weather)

    # flawlessly serialize the finished clean artifact to the defined physical drive structure
    #
    daily.to_csv(os.path.join(OUT_DAILY, f"{uid}_daily.csv"), index=False)

    # provide a simple console log strictly for manual tracking progress
    #
    print(f"PROCESSED AND SERIALIZED: {uid}_daily.csv")


INITIATING MASSIVE PIPELINE EXECUTION FOR ALL FILES
PROCESSED AND SERIALIZED: timeseries_table_timeseries_table (1)_daily.csv
PROCESSED AND SERIALIZED: timeseries_table_timeseries_table (10)_daily.csv
PROCESSED AND SERIALIZED: timeseries_table_timeseries_table (100)_daily.csv
PROCESSED AND SERIALIZED: timeseries_table_timeseries_table (11)_daily.csv
PROCESSED AND SERIALIZED: timeseries_table_timeseries_table (12)_daily.csv
PROCESSED AND SERIALIZED: timeseries_table_timeseries_table (13)_daily.csv
PROCESSED AND SERIALIZED: timeseries_table_timeseries_table (14)_daily.csv
PROCESSED AND SERIALIZED: timeseries_table_timeseries_table (15)_daily.csv
PROCESSED AND SERIALIZED: timeseries_table_timeseries_table (16)_daily.csv
PROCESSED AND SERIALIZED: timeseries_table_timeseries_table (17)_daily.csv
PROCESSED AND SERIALIZED: timeseries_table_timeseries_table (18)_daily.csv
PROCESSED AND SERIALIZED: timeseries_table_timeseries_table (19)_daily.csv
PROCESSED AND SERIALIZED: timeseries_table_time

In [ ]:
"""
=========================================================================================
BLOCK 12: DATA RECONSTRUCTION AND ANTI-LEAKAGE LAG GENERATION
=========================================================================================
This block ensures the pipeline is self-contained. It re-loads the daily CSVs,
enforces the calendar grid, and generates the 7-day memory lags.
=========================================================================================
"""

# import required libraries for file discovery and data manipulation
#
import os
import glob
import numpy as np
import pandas as pd

# define the local output directory containing the processed daily files
#
LOCAL_OUT_DAILY = "/content/drive/MyDrive/raw_indoor_plus_outdoor/processed_daily"

# define the number of historical lag days to generate for the memory features
#
LAG_DAYS = 7

# discover all processed daily csv files and sort them chronologically
#
daily_paths = sorted(glob.glob(os.path.join(LOCAL_OUT_DAILY, "*.csv")))

# mathematically concatenate all isolated daily files into a single master dataframe
#
raw_master = pd.concat([pd.read_csv(f) for f in daily_paths], ignore_index=True)

# strictly force the date column to pandas datetime to ensure safe physical merging
#
raw_master["Date"] = pd.to_datetime(raw_master["Date"])

# identify columns that explicitly track ping frequency to prevent mathematical target leakage
#
leakage_cols = [c for c in raw_master.columns if "count_nonnull" in c or "ping_count" in c]

# violently purge the identified leakage columns from the master dataset
#
raw_master = raw_master.drop(columns=leakage_cols, errors="ignore")

def local_grid(grp):
    """
    function: local_grid
    arguments: grp (pandas DataFrame group isolated to a single equipment ID)
    return: grp (pandas DataFrame strictly reindexed to a continuous daily calendar)
    description: Enforces a chronological grid by injecting completely empty
    rows for any physically missing days, preventing time-travel leakage during lag generation.
    """
    # generate a perfect, continuous daily calendar grid spanning the unit's active lifetime
    #
    dr = pd.date_range(start=grp["Date"].min(), end=grp["Date"].max(), freq='D')

    # reindex the isolated group strictly against the continuous calendar to inject missing days
    #
    grp = grp.set_index("Date").reindex(dr)

    # persist the unit identifier across the newly injected empty calendar days
    #
    grp["Equipment_ID"] = grp["Equipment_ID"].ffill().bfill()

    # cleanly return the perfectly gridded temporal matrix
    #
    return grp.rename_axis("Date").reset_index()

# execute the calendar gridding logic grouped strictly by mechanical unit identifier
#
master_processed = raw_master.groupby("Equipment_ID", group_keys=False).apply(local_grid)

# algebraically calculate the mathematical nullity ratio for every column in the matrix
#
missing_map = master_processed.isnull().mean()

# aggressively drop any feature that is missing more than 95 percent of its physical observations
#
master_processed = master_processed.drop(columns=missing_map[missing_map > 0.95].index.tolist())

# isolate strictly the numeric features required for continuous mathematical lag generation
#
num_cols = master_processed.select_dtypes(include=[np.number]).columns.tolist()

# initialize an empty array to temporarily store the generated lag sequences
#
lags = []

# iteratively generate historical lag steps up to the defined memory threshold
#
for i in range(1, LAG_DAYS + 1):

    # mathematically shift the numeric columns downward by the specific day interval
    #
    s = master_processed.groupby("Equipment_ID")[num_cols].shift(i)

    # explicitly rename the shifted columns to perfectly track their temporal depth
    #
    s.columns = [f"{c}_lag_{i}" for c in s.columns]

    # append the successfully generated lag step to the tracking array
    #
    lags.append(s)

# securely concatenate the original matrix with the generated historical lag matrices
#
master_local_lags = pd.concat([master_processed] + lags, axis=1)

# explicitly cast the equipment identifier to a categorical datatype for optimal model routing
#
master_local_lags["Equipment_ID"] = master_local_lags["Equipment_ID"].astype("category")

# mathematically isolate the generated historical memory features
#
lag_feats = [c for c in master_local_lags.columns if "_lag_" in c]

# explicitly isolate the current-day environmental and seasonal features
#
day_t_feats = [c for c in num_cols if any(x in c for x in ["outdoor", "outside", "true_", "humidity", "month", "day_of_week", "is_weekend"])]

# compile the final strictly engineered feature set merging historical memory with current environment
#
final_feature_set = lag_feats + day_t_feats

# cleanly report the successful generation and dimensionality of the mathematical feature matrix
#
print(f"DATA READY. TOTAL FEATURES DEFINED: {len(final_feature_set)}")

/tmp/ipykernel_3283/1520631487.py:71: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  master_processed = raw_master.groupby("Equipment_ID", group_keys=False).apply(local_grid)


DATA READY. TOTAL FEATURES DEFINED: 457


In [ ]:
print(lag_feats)
print(day_t_feats)

['daily_heating_hours_lag_1', 'daily_cooling_hours_lag_1', 'daily_off_hours_lag_1', 'daily_unknown_hours_lag_1', 'daily_runtime_hours_lag_1', 'total_included_known_mode_hours_lag_1', 'total_unknown_hours_lag_1', 'daily_fan_on_hours_lag_1', 'fan_runtime_ratio_lag_1', 'setpoint_change_count_lag_1', 'daily_idle_hours_lag_1', 'daily_heat_high_hours_lag_1', 'daily_heat_med_hours_lag_1', 'daily_heat_low_hours_lag_1', 'daily_pre_heat_hours_lag_1', 'daily_post_heat_hours_lag_1', 'daily_refresh_heat_hours_lag_1', 'daily_cool_high_hours_lag_1', 'daily_cool_med_hours_lag_1', 'daily_cool_low_hours_lag_1', 'daily_pre_cool_hours_lag_1', 'daily_post_cool_hours_lag_1', 'daily_refresh_cool_hours_lag_1', 'indoor_temp_time_weighted_mean_lag_1', 'setpoint_time_weighted_mean_lag_1', 'setpoint_gap_mean_lag_1', 'day_of_week_lag_1', 'is_weekend_lag_1', 'month_lag_1', 'month_sin_lag_1', 'month_cos_lag_1', 'indoor_temp_min_lag_1', 'indoor_temp_q25_lag_1', 'indoor_temp_median_lag_1', 'indoor_temp_q75_lag_1', 'in

In [ ]:
"""
=========================================================================================
BLOCK 13: APPROACH 1 - INDIVIDUAL LOCALIZED MODELS (PER-HOUSE TRAINING)
=========================================================================================
This block treats every house as an isolated thermodynamic entity. It iterates through
each Equipment_ID, splits the data chronologically using datetime,
trains a LightGBM model to naturally handle the NaNs, and evaluates the pooled physical error.
It additionally flags any architecture that mathematically collapses (R2 < 0).
=========================================================================================
"""

from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

local_params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'num_leaves': 15,
    'max_depth': 5,
    'min_child_samples': 10,
    'random_state': 27,
    'n_jobs': -1,
    'verbose': -1
}

# initialize aggregate tracking arrays to pool true physical values and predictions
#
pooled_y_true = []
pooled_y_pred = []
successful_models = 0

# initialize an explicit tracking array to capture failed architectures
#
flagged_houses = []

for house_id in master_local_lags["Equipment_ID"].unique():

    house_df = master_local_lags[master_local_lags["Equipment_ID"] == house_id].copy()
    house_df = house_df.dropna(subset=["daily_runtime_hours"]).sort_values("Date")

    if len(house_df) < 60:
        continue

    # define the absolute chronological threshold dividing the past and the future
    #
    split_barrier = house_df["Date"].quantile(0.8)
    train_mask = house_df["Date"] <= split_barrier

    # mathematically enforce chronological splitting
    #
    X_train_local = house_df[train_mask][final_feature_set]
    y_train_local = house_df[train_mask]["daily_runtime_hours"]

    X_test_local = house_df[~train_mask][final_feature_set]
    y_test_local = house_df[~train_mask]["daily_runtime_hours"]

    local_model = LGBMRegressor(**local_params)
    local_model.fit(X_train_local, y_train_local)

    preds = local_model.predict(X_test_local)

    # calculate the isolated coefficient of determination for this specific house
    #
    local_r2 = r2_score(y_test_local, preds)

    # strictly flag the unit if the model performs mathematically worse than a naive mean
    #
    if local_r2 < 0.0:
        flagged_houses.append(house_id)

    # strictly append the raw physical predictions and targets to the pooled tracking arrays
    #
    pooled_y_true.extend(y_test_local.values)
    pooled_y_pred.extend(preds)

    # increment the tracking counter to audit total successful isolated architectures
    #
    successful_models += 1

# evaluate the unified physics metrics across all pooled localized predictions
#
pooled_rmse = root_mean_squared_error(pooled_y_true, pooled_y_pred)
pooled_mae = mean_absolute_error(pooled_y_true, pooled_y_pred)
pooled_r2 = r2_score(pooled_y_true, pooled_y_pred)

print(f"Successfully trained {successful_models} localized models.")
print(f"Flagged {len(flagged_houses)} models for (R2 < 0).")
print(f"POOLED LOCAL RMSE: {pooled_rmse:.4f} Hours")
print(f"POOLED LOCAL MAE:  {pooled_mae:.4f} Hours")
print(f"POOLED LOCAL R2:   {pooled_r2:.4f}")

Successfully trained 100 localized models.
Flagged 26 models for (R2 < 0).
POOLED LOCAL RMSE: 4.1610 Hours
POOLED LOCAL MAE:  3.0518 Hours
POOLED LOCAL R2:   0.4990


In [ ]:
"""
=========================================================================================
BLOCK 14: APPROACH 2 - GLOBAL PHYSICS MODEL (PURE GENERALIZATION)
=========================================================================================
This block treats all houses as a shared thermodynamic dataset. It splits the global
timeline chronologically, training the model strictly on universal physics (weather
and lags).
=========================================================================================
"""

# explicitly drop rows where the physical prediction target is missing across all data
#
global_df = master_local_lags.dropna(subset=["daily_runtime_hours"]).copy()

# rigorously sort the massive dataframe strictly by calendar date
#
global_df = global_df.sort_values("Date").reset_index(drop=True)

# calculate the absolute datetime barrier mathematically to explicitly prevent temporal bleeding
#
split_barrier = global_df["Date"].quantile(0.8)

# explicitly log the global time split barrier
#
print(f"GLOBAL TIME SPLIT BARRIER: {split_barrier.strftime('%Y-%m-%d')}")

# mathematically isolate the training matrix strictly before the chronological barrier
#
train_mask = global_df["Date"] <= split_barrier

# enforce strict chronological separation for the training features excluding hardware ids
#
X_train_global = global_df[train_mask][final_feature_set]

# enforce strict chronological separation for the training target
#
y_train_global = global_df[train_mask]["daily_runtime_hours"]

# isolate the testing matrix strictly after the chronological barrier excluding hardware ids
#
X_test_global = global_df[~train_mask][final_feature_set]

# enforce strict chronological separation for the testing target
#
y_test_global = global_df[~train_mask]["daily_runtime_hours"]

# define parameters meant for heavy massive-scale tree structures
#
global_params = {
    'n_estimators': 800,
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': 8,
    'min_child_samples': 50,
    'colsample_bytree': 0.8,
    'random_state': 27,
    'n_jobs': -1,
    'verbose': -1
}

# perfectly instantiate the global machine learning architect
#
global_model = LGBMRegressor(**global_params)

# strictly map the pure thermodynamic features during global training
#
global_model.fit(X_train_global, y_train_global)

# generate pure blind predictions against the global future timeline
#
global_preds = global_model.predict(X_test_global)

# extract explicit physical validation metrics
#
g_rmse = root_mean_squared_error(y_test_global, global_preds)
g_mae = mean_absolute_error(y_test_global, global_preds)
g_r2 = r2_score(y_test_global, global_preds)

# outside tempature changes, how much did the tempature change outside
# over the last few days
#
# last days runtime
#
# take mean outdoor tempature over last n days, do a linear fit, find its linearized
# gradient
#
# formally report the global outcome
#
# rfe/mutual information
#
# cluster household with similar behviors. one model per cluster,
# relies on the assumption that clusters exist
# https://tslearn.readthedocs.io/en/stable/user_guide/clustering.html
#
# make sure the daily statistics (variance, etc) is weighted
#
print(f"GLOBAL RMSE: {g_rmse:.4f} HOURS")
print(f"GLOBAL MAE:  {g_mae:.4f} HOURS")
print(f"GLOBAL R2:   {g_r2:.4f}")

GLOBAL TIME SPLIT BARRIER: 2026-01-01
GLOBAL RMSE: 3.2299 HOURS
GLOBAL MAE:  2.4182 HOURS
GLOBAL R2:   0.6959


In [ ]:
"""
=========================================================================================
BLOCK 14C: GLOBAL NEURAL NETWORK EXPERIMENT (PURE GENERALIZATION)
=========================================================================================
This block treats all houses as a shared dataset for a Multi-Layer Perceptron (NN).
It strictly excludes Equipment_ID to ensure universal generalization.
To satisfy the rigid mathematical requirements of neural networks, it performs:
1. Column Filtering: Drops features that are >95% NaN across the global dataset.
2. Row Filtering: Discards any remaining row containing even a single NaN.
3. Feature Scaling: Standardizes the input matrix to ensure gradient stability.
=========================================================================================
"""

# import the multilayer perceptron regressor from scikit-learn
#
from sklearn.neural_network import MLPRegressor

# import the standard scaler to normalize features for the neural network
#
from sklearn.preprocessing import StandardScaler

# initialize the neural network with a stable, large-scale solver
#
nn_global_params = {
    'hidden_layer_sizes': (100, 50),
    'activation': 'relu',
    'solver': 'adam',
    'alpha': 0.001,
    'max_iter': 500,
    'random_state': 27,
    'early_stopping': True,
    'validation_fraction': 0.1
}

# create a copy of the master matrix and drop features that are almost entirely empty
#
rigid_global_df = master_local_lags.copy()
mostly_null_cols = rigid_global_df.columns[rigid_global_df.isnull().mean() > 0.95]
rigid_global_df = rigid_global_df.drop(columns=mostly_null_cols)

# strictly define the feature set excluding hardware identifiers to force generalization
#
nn_global_features = [f for f in final_feature_set if f in rigid_global_df.columns]

# execute a violent row-drop to remove every day that has even a single missing sensor value
#
rigid_global_df = rigid_global_df.dropna(subset=["daily_runtime_hours"] + nn_global_features).copy()

# rigorously sort by date to prepare for the chronological split
#
rigid_global_df = rigid_global_df.sort_values("Date").reset_index(drop=True)

# calculate the split barrier based on the 80th percentile of the surviving data timeline
#
split_barrier_nn = rigid_global_df["Date"].quantile(0.8)

# isolate the training and testing sets based on the chronological barrier
#
train_mask_nn = rigid_global_df["Date"] <= split_barrier_nn
X_train_nn = rigid_global_df[train_mask_nn][nn_global_features]
y_train_nn = rigid_global_df[train_mask_nn]["daily_runtime_hours"]
X_test_nn = rigid_global_df[~train_mask_nn][nn_global_features]
y_test_nn = rigid_global_df[~train_mask_nn]["daily_runtime_hours"]

# instantiate and apply the scaler to the training past and transform the testing future
#
scaler_nn = StandardScaler()
X_train_scaled = scaler_nn.fit_transform(X_train_nn)
X_test_scaled = scaler_nn.transform(X_test_nn)

# perfectly instantiate and train the global neural network architect
#
global_nn_model = MLPRegressor(**nn_global_params)
global_nn_model.fit(X_train_scaled, y_train_nn)

# generate predictions and clip them to the physical 24-hour limit
#
nn_preds = np.clip(global_nn_model.predict(X_test_scaled), 0, 24)

# extract and report the final physical validation metrics
#
nn_rmse = root_mean_squared_error(y_test_nn, nn_preds)
nn_mae = mean_absolute_error(y_test_nn, nn_preds)
nn_r2 = r2_score(y_test_nn, nn_preds)

# formally report the findings of the generalized neural network experiment
#
print(f"HOUSES UTILIZED IN GLOBAL NN: {rigid_global_df['Equipment_ID'].nunique()}")
print(f"GLOBAL TIME SPLIT BARRIER: {split_barrier_nn.strftime('%Y-%m-%d')}")
print(f"GLOBAL NN RMSE: {nn_rmse:.4f} HOURS")
print(f"GLOBAL NN MAE:  {nn_mae:.4f} HOURS")
print(f"GLOBAL NN R2:   {nn_r2:.4f}")


--- INITIATING GLOBAL NEURAL NETWORK TRAINING (APPROACH 3) ---
Houses utilized in global NN: 98
Global Time Split Barrier: 2026-01-08
GLOBAL NN RMSE: 3.5889 Hours
GLOBAL NN MAE:  2.6180 Hours
GLOBAL NN R2:   0.6445
